# Limpieza del dataset

Los datos utilizados vienen del dataset de Kaggle [Blackjack Dataset](https://www.kaggle.com/datasets/fry1999/blackjack-dataset). Este contiene un total de 50 millones de filas con información sobre partidas entre un jugador y el crupier (o *dealer*), como la mano con la que empieza el jugador, la mano con la que termina, el valor total de la mano final, las acciones realizadas a lo largo de la partida, además de información sobre las cartas del *dealer* e información general de la partida.

El objetivo de esta limpieza es obtener un dataset limpio separando las filas por cada acción de la columna `actions_taken`, dado que el modelo a desarrollar debe ser capaz de recomendar la siguiente acción en base a la mano del jugador y la carta del crupier. Por ejemplo:

Con la fila **initial_hand** [3, 9] | **player_final** [[3, 9, 4, 4]] | **actions_taken** ['H', 'H', 'S'] se obtendrían las siguientes filas resultantes:
| **player_hand** | **action** |
| --------------- | ---------- |
| [3, 9]          |   ['H']    |
| [3, 9, 4]       |   ['H']    |
| [3, 9, 4, 4]    |   ['S']    |

Para una fila con *splits* sería de la siguiente manera: **initial_hand** [3, 3] | **player_final** [[3, 2, 10], [3, 4, 5]] | **actions_taken** [['P', 'H', 'S'], ['H', 'S']]

Filas resultantes:
| **player_hand** | **action** |
| --------------- | ---------- |
| [3, 3]          |   ['P']    |
| [3, 2]          |   ['H']    |
| [3, 2, 10]      |   ['S']    |
| [3, 4]          |   ['H']    |
| [3, 4, 5]       |   ['S']    |

In [1]:
import pandas as pd
import zipfile_deflate64 as zipfile
from sklearn.preprocessing import LabelEncoder

### Carga de datos

Para cargar los datos los importamos del zip y seleccionamos una fracción aleatoria del 10%, dado que las 50 millones de filas resultan en demasiado tiempo de preprocesado y de entrenamiento. Tambien excluimos aquellas filas en las que la acción está vacía, ya que esa será nuestra variable objetivo.

In [2]:
# Carga de datos
zf = zipfile.ZipFile("../BlackjackData/blackjack_dataset.zip") 
df_raw = pd.read_csv(zf.open("blackjack_dataset.csv"))

In [3]:
df = df_raw[~df_raw['actions_taken'].str.contains('\\[]')].sample(frac=0.1, random_state=21)
df.head(15)

,shoe_id,cards_remaining,dealer_up,initial_hand,dealer_final,dealer_final_value,player_final,player_final_value,actions_taken,run_count,true_count,win
9895771,162845,249,2,"[2, 6]","[2, 10, 5]",17,"[[2, 6, 5]]",[13],"[['H', 'S']]",-4,0,-1.0
3095864,50943,148,7,"[10, 10]","[7, 4, 9]",20,"[[10, 10]]",[20],[['S']],-3,-1,0.0
5996596,98679,198,9,"[5, 8]","[9, 11]",20,"[[5, 8, 11, 8]]",[22],"[['H', 'H']]",3,0,-1.0
41522709,683341,346,9,"[5, 2]","[9, 10]",19,"[[5, 2, 6, 6]]",[19],"[['H', 'H', 'S']]",9,1,0.0
45214930,744106,150,6,"[10, 9]","[6, 10, 8]",24,"[[10, 9]]",[19],[['S']],6,2,1.0
24272846,399453,394,4,"[8, 10]","[4, 10, 5]",19,"[[8, 10]]",[18],[['S']],0,0,-1.0
48288070,794672,344,3,"[5, 10]","[3, 10, 8]",21,"[[5, 10]]",[15],[['S']],-10,-1,-1.0
9193404,151288,165,4,"[10, 10]","[4, 9, 10]",23,"[[10, 10]]",[20],[['S']],-7,-2,1.0
13842927,227797,388,2,"[10, 10]","[2, 10, 5]",17,"[[10, 10]]",[20],[['S']],-1,0,1.0
17299790,284697,340,10,"[4, 9]","[10, 2, 11, 8]",21,"[[4, 9, 11, 10]]",[24],"[['H', 'H']]",-6,0,-1.0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4854086 entries, 9895771 to 39814164
Data columns (total 12 columns):
 #   Column              Dtype  
---  ------              -----  
 0   shoe_id             int64  
 1   cards_remaining     int64  
 2   dealer_up           int64  
 3   initial_hand        object 
 4   dealer_final        object 
 5   dealer_final_value  object 
 6   player_final        object 
 7   player_final_value  object 
 8   actions_taken       object 
 9   run_count           int64  
 10  true_count          int64  
 11  win                 float64
dtypes: float64(1), int64(5), object(6)
memory usage: 481.4+ MB


In [5]:
df.isna().sum()

shoe_id               0
cards_remaining       0
dealer_up             0
initial_hand          0
dealer_final          0
dealer_final_value    0
player_final          0
player_final_value    0
actions_taken         0
run_count             0
true_count            0
win                   0
dtype: int64

### Preprocesado del dataset

Para realizar el preprocesado primero limpiamos ciertas columnas y arreglamos el formato de otras. La columna `win` (que más adelante será descartada) la reducimos a tres opciones, empatar, ganar y perder; y las columnas `initial_hand`, `player_final` y `actions_taken` que son de tipo `string` originalmente, las convertimos a listas para poder iterar sobre ellas.

In [6]:
df["win"] = df["win"].apply(lambda x: 0 if x == 0 else 1 if x < 0 else 2) # 0 empate, 1 pierde, 2 gana

def str_to_list(cell, toInt = True):
    if "[[" in cell: cell = cell[1:-1]
    res = []
    for ls in cell.split("],"):
        ls = ''.join(c for c in ls if c not in "'[]").split(', ')
        
        if toInt: ls = list(map(int, ls))
        res.append(ls)

    if len(res) > 1:
        res = res
    else:
        res = res[0]
    
    return res

df["initial_hand"] = df["initial_hand"].apply(lambda x: str_to_list(x))
df["player_final"] = df["player_final"].apply(lambda x: str_to_list(x))
df["actions_taken"] = df["actions_taken"].apply(lambda x: str_to_list(x, False))

Tras ajustar los tipos de las columnas, podemos continuar con la separación de las filas por cada acción realizada en la partida a la que corresponden. Al hacer este cambio nos quedamos solo con los casos en los que se ha hecho un solo *split*, dado que si se hacía más de uno, aumentaba la complejidad del preprocesado considerablemente y por como se va a realizar el modelo de *Computer Vision* y Blackjack completo, no eran del todo relevantes.

In [7]:
def create_row(df_row, hand, action = ""):
    res_row = []

    # Añadimos primera acción con initial_hand
    res_row.append(df_row[0]) # shoe_id
    res_row.append(df_row[1]) # cards_remaining
    res_row.append(df_row[2]) # dealer_up

    res_row.append(hand.copy()) # initial_hand -> player_hand
    if (11 in hand and sum(hand) > 21):
        for i in range(0, len(hand)):
            if hand[i] == 11 and sum(hand) > 21:
                hand[i] = 1
    res_row.append(sum(hand)) # player_final_value -> player_final_action_value

    if action == "":
        if type(df_row[6][0]) == str:
            res_row.append(str(df_row[6][0]).strip()) # actions_taken, first_action -> action
        else:
            res_row.append(str(df_row[6][0][0]).strip())
    else:
        res_row.append(str(action).strip())

    res_row.append(df_row[7]) # run_count
    res_row.append(df_row[8]) # true_count
    res_row.append(df_row[9]) # win

    return res_row

In [8]:
original_cols = ["shoe_id", "cards_remaining", "dealer_up", "initial_hand", "player_final", "player_final_value", "actions_taken", "run_count", "true_count", "win"]
#                   0     |         1        |      2     |       3       |       4       |          5          |        6       |      7     |       8     |   9
df_list = df[original_cols].values.tolist()

new_df_list = []
for row in df_list:
    # Añadimos primera acción con initial_hand
    initial_row = create_row(row, row[3])
    new_df_list.append(initial_row)

    aux_row = initial_row
    if (type(row[4][0]) == int): # si no se ha hecho split
        for h, a in zip(row[4][2:], row[6][1:]):
            if "D" not in aux_row[5]:
                new_row = create_row(row, aux_row[3] + [h], a)
                aux_row = new_row

                new_df_list.append(new_row)
    else:
        if len(row[4]) <= 2: # Solo nos quedamos filas con un solo split
            for i, (split_hand, split_action) in enumerate(zip(row[4], row[6])):
                if "D" not in split_action[0]:
                    # Primera mano del split
                    if i == 0:
                        initial_split_row = create_row(row, split_hand[:2], split_action[1])
                        new_df_list.append(initial_split_row)

                        aux_row_split = initial_split_row
                        for h, a in zip(split_hand[2:], split_action[2:]):
                            if "D" not in aux_row_split[5]:
                                new_split_row = create_row(row, aux_row_split[3] + [h], a)
                                aux_row_split = new_split_row

                                new_df_list.append(new_split_row)
                    else:
                        initial_split_row = create_row(row, split_hand[:2], split_action[0])
                        new_df_list.append(initial_split_row)

                        aux_row_split = initial_split_row
                        for h, a in zip(split_hand[2:], split_action[1:]):
                            if "D" not in aux_row_split[5]:
                                new_split_row = create_row(row, aux_row_split[3] + [h], a)
                                aux_row_split = new_split_row

                                new_df_list.append(new_split_row)


Finalmente, transformamos la lista de filas obtenidas del preprocesado a un dataset nuevo. También separamos la lista de cartas de la columna `player_hand` en columnas individuales, para evitar tener problemas con listas o `strings` al entrenar el modelo.

In [9]:
df_clean = pd.DataFrame(new_df_list)
df_clean.columns =["shoe_id", "cards_remaining", "dealer_up", "player_hand", "player_hand_value", "action", "run_count", "true_count", "win"]
df_clean

,shoe_id,cards_remaining,dealer_up,player_hand,player_hand_value,action,run_count,true_count,win
0,162845,249,2,"[2, 6]",8,H,-4,0,1
1,162845,249,2,"[2, 6, 5]",13,S,-4,0,1
2,50943,148,7,"[10, 10]",20,S,-3,-1,0
3,98679,198,9,"[5, 8]",13,H,3,0,1
4,98679,198,9,"[5, 8, 11]",14,H,3,0,1
...,...,...,...,...,...,...,...,...,...
6930328,678580,408,4,"[10, 5]",15,S,-8,-1,2
6930329,503891,393,8,"[6, 3]",9,H,0,0,1
6930330,503891,393,8,"[6, 3, 5]",14,H,0,0,1
6930331,245594,270,11,"[10, 9]",19,N,-5,0,1


In [12]:
df_cartas = pd.DataFrame(df_clean["player_hand"].tolist(), index=df_clean.index)
df_cartas.columns = [f"carta_{i+1}" for i in range(df_cartas.shape[1])]
df_cartas = df_cartas.fillna(0).astype(int)
df_clean = pd.concat([df_clean, df_cartas], axis=1).drop(columns=["player_hand"])

Exportamos este nuevo dataset a un fichero para poder accederlo más facilmente y además seleccionamos un 10% de este dataset para usar como datos de validación, tambien exportandolos a otro fichero distinto. Además, al exportarlos, lo hacemos sin incluir las columnas `shoe_id` y `win`, dado que no aportan ningun valor al modelo.

In [13]:
df_clean_split = df_clean.drop(columns=["shoe_id", "win"]).sample(frac=0.9, random_state=21)
df_clean_val = df_clean.drop(columns=["shoe_id", "win"]).loc[~df_clean.index.isin(df_clean_split.index)].reset_index(drop=True)
df_clean_split = df_clean_split.reset_index(drop=True)

df_clean_split.to_csv("../BlackjackData/blackjack_dataset_clean.csv")
df_clean_val.to_csv("../BlackjackData/blackjack_dataset_clean_validation.csv")